In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

recommendations = spark.table(
    "agentdb.intelligence.recommendation_registry"
)

inventory_risk = spark.table(
    "agentdb.intelligence.inventory_risk"
)

In [0]:
evaluation_base = (

    recommendations.alias("r")

    .join(

        inventory_risk.alias("i"),

        [
            "product_key",
            "store_key"
        ]
    )
)

In [0]:
evaluation_base = (

    evaluation_base

    .withColumn(

        "would_prevent_stockout",

        when(

            (
                col(
                    "recommendation_type"
                ) == "REORDER"
            )

            &

            (
                col(
                    "projected_days_to_stockout"
                ) < 30
            ),

            1
        )

        .otherwise(0)
    )
)

In [0]:
evaluation = (

    evaluation_base

    .agg(

        count("*")
        .alias(
            "total_recommendations"
        ),

        sum(

            when(
                col(
                    "recommendation_type"
                ) == "REORDER",
                1
            )

        ).alias(
            "reorder_recommendations"
        ),

        sum(

            when(
                col(
                    "recommendation_type"
                ) == "EXPEDITE_PO",
                1
            )

        ).alias(
            "expedite_recommendations"
        ),

        sum(

            when(
                col(
                    "recommendation_type"
                ) == "SUPPLIER_ALERT",
                1
            )

        ).alias(
            "supplier_alerts"
        ),

        sum(
            "would_prevent_stockout"
        ).alias(
            "stockouts_prevented"
        )
    )
)

In [0]:
evaluation = (

    evaluation

    .withColumn(

        "false_positive_recommendations",

        col(
            "total_recommendations"
        )

        -

        col(
            "stockouts_prevented"
        )
    )

    .withColumn(

        "precision",

        col(
            "stockouts_prevented"
        )

        /

        col(
            "total_recommendations"
        )
    )

    .withColumn(
        "recall",
        lit(None).cast("double")
    )

    .withColumn(
        "evaluation_date",
        current_date()
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

In [0]:
(
    evaluation

    .write

    .mode("overwrite")

    .saveAsTable(
        "agentdb.evaluation.recommendation_metrics"
    )
)

In [0]:
%sql
SELECT * FROM agentdb.evaluation.recommendation_metrics
LIMIT 10